# Spike-Based ALU — Turing-Complete Computation with Spikes

**SC-NeuroCore v3.14** — Beyond pattern recognition: general-purpose computation.

Spiking neural networks are usually framed as pattern classifiers.
SC-NeuroCore breaks this paradigm by implementing a full arithmetic
logic unit using only spike-based logic gates.

1. **Logic gates** — AND, OR, NOT, NAND, XOR as LIF threshold operations
2. **SR latch register** — bistable neuron pairs store bits
3. **Ripple-carry adder** — spike-based binary addition
4. **ALU operations** — add, subtract, bitwise ops, compare
5. **Spike sort** — comparison network for sorting integers

All operations use LIF neurons. No floating-point, no conventional
logic — purely neuromorphic computation.

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore.symbolic.spike_logic import (
    SpikeGate,
    SpikeRegister,
    SpikeALU,
    spike_sort,
)

print("SC-NeuroCore spike ALU demo")

## 1. Spike-Based Logic Gates

Each gate is a LIF neuron with carefully chosen threshold and
weights. The mapping between logic and neuroscience:

| Gate | Threshold | Weights | LIF interpretation |
|------|-----------|---------|--------------------|
| AND | 2 | [1, 1] | Fires only when both inputs spike |
| OR | 1 | [1, 1] | Fires when any input spikes |
| NOT | 0 | [-1] | Fires when input is silent |
| NAND | 0 | [-1, -1] + bias=2 | Inhibition from both silences output |
| XOR | 1 | [1, 1] + mutual inhibit | Fires on exactly one input |

In [ ]:
gates = ["AND", "OR", "NOT", "NAND", "XOR"]

print(f"{'Gate':<6s}  {'Truth Table':>40s}")
print("-" * 48)

for gate_name in gates:
    gate = SpikeGate(gate_type=gate_name)
    if gate_name == "NOT":
        results = [f"{a}→{gate(a)}" for a in [0, 1]]
    else:
        results = [f"({a},{b})→{gate(a,b)}" for a in [0, 1] for b in [0, 1]]
    print(f"{gate_name:<6s}  {', '.join(results):>40s}")

# Verify against Python logic
and_gate = SpikeGate("AND")
or_gate = SpikeGate("OR")
not_gate = SpikeGate("NOT")
xor_gate = SpikeGate("XOR")

for a in [0, 1]:
    for b in [0, 1]:
        assert and_gate(a, b) == (a & b)
        assert or_gate(a, b) == (a | b)
        assert xor_gate(a, b) == (a ^ b)
    assert not_gate(a) == (1 - a)

print("\nAll gates verified against Python logic.")

## 2. Spike Register (SR Latch)

Each bit is stored by a pair of mutually inhibiting neurons
(bistable). Write injects a spike to set or reset; read checks
which neuron is active. This is the spiking equivalent of a
flip-flop.

In [ ]:
reg = SpikeRegister(n_bits=8)

test_values = [0, 42, 127, 255, 170, 85]

print(f"{'Written':>8s}  {'Read':>8s}  {'Binary':>10s}  {'Match':>6s}")
print("-" * 36)
for val in test_values:
    reg.write(val)
    read_val = reg.read()
    binary = format(val, "08b")
    match = "ok" if read_val == val else "FAIL"
    print(f"{val:8d}  {read_val:8d}  {binary:>10s}  {match:>6s}")

# Bit-level access
reg.write(0b10110011)
bits = reg.read_bits()
print(f"\nBit-level read of 0b10110011: {bits}")

## 3. Spike ALU

The `SpikeALU` composes spike gates into a ripple-carry adder,
two's complement subtractor, bitwise operations, and comparator.
All arithmetic is 8-bit unsigned.

In [ ]:
alu = SpikeALU(n_bits=8)

# Addition
print("=== Addition ===")
add_tests = [(10, 20), (100, 155), (0, 0), (127, 128), (1, 1)]
for a, b in add_tests:
    result, carry = alu.add(a, b)
    expected = (a + b) & 0xFF
    c_exp = (a + b) > 255
    status = "ok" if result == expected and carry == c_exp else "FAIL"
    print(f"  {a:3d} + {b:3d} = {result:3d} (carry={int(carry)})  [{status}]")

# Subtraction
print("\n=== Subtraction ===")
sub_tests = [(50, 20), (0, 1), (255, 255), (100, 99)]
for a, b in sub_tests:
    result, borrow = alu.sub(a, b)
    expected = (a - b) & 0xFF
    status = "ok" if result == expected else "FAIL"
    print(f"  {a:3d} - {b:3d} = {result:3d} (borrow={int(borrow)})  [{status}]")

# Bitwise XOR
print("\n=== Bitwise XOR ===")
xor_tests = [(0xFF, 0x00), (0xAA, 0x55), (42, 42), (0, 255)]
for a, b in xor_tests:
    result = alu.bitwise_xor(a, b)
    expected = a ^ b
    status = "ok" if result == expected else "FAIL"
    print(f"  0x{a:02X} ^ 0x{b:02X} = 0x{result:02X}  [{status}]")

# Compare
print("\n=== Compare ===")
cmp_tests = [(10, 20), (20, 10), (15, 15)]
for a, b in cmp_tests:
    result = alu.compare(a, b)
    expected = -1 if a < b else (1 if a > b else 0)
    status = "ok" if result == expected else "FAIL"
    symbols = {-1: "<", 0: "=", 1: ">"}
    print(f"  {a:3d} {symbols[result]} {b:3d}  [{status}]")

## 4. Spike Sort

`spike_sort()` implements a comparison network using
`SpikeALU.compare()`. Bubble-sort topology: O(n²) comparisons,
each using spike gates.

In [ ]:
test_arrays = [
    [5, 3, 8, 1, 9, 2, 7, 4, 6],
    [255, 0, 128, 64, 192],
    [42],
    [10, 10, 10, 10],
    list(range(15, 0, -1)),
]

for arr in test_arrays:
    sorted_arr = spike_sort(arr)
    expected = sorted(arr)
    match = "ok" if sorted_arr == expected else "FAIL"
    print(f"  {arr} → {sorted_arr}  [{match}]")

## 5. Gate Cost Analysis

Each spike gate maps to a single LIF neuron. The ALU cost
scales linearly with bit width.

In [ ]:
bit_widths = [4, 8, 16, 32]

print(f"{'Bits':>5s}  {'Add neurons':>12s}  {'Sub neurons':>12s}  "
      f"{'XOR neurons':>12s}  {'CMP neurons':>12s}")
print("-" * 58)
for n in bit_widths:
    # Full adder: 2 XOR + 2 AND + 1 OR = 5 gates per bit + carry chain
    add_neurons = 5 * n + 1  # ripple carry
    sub_neurons = add_neurons + n  # NOT for two's complement
    xor_neurons = n  # 1 XOR gate per bit
    cmp_neurons = 3 * n  # subtract + sign check
    print(f"{n:5d}  {add_neurons:12d}  {sub_neurons:12d}  "
          f"{xor_neurons:12d}  {cmp_neurons:12d}")

# Compare to FPGA LUT cost
print("\nFor reference: on Xilinx Artix-7, each LIF neuron ≈ 30-50 LUTs.")
print(f"8-bit spike ALU: ~{(5*8+1+8+8+3*8) * 40:,} LUTs (very rough estimate).")
print("An 8-bit conventional ALU: ~50-100 LUTs. Spike ALU is ~100× larger.")
print("The value is not efficiency — it is that computation emerges from")
print("the SAME substrate that does perception and learning.")

## Summary

| Component | Spike implementation | Neuron count (8-bit) |
|-----------|---------------------|---------------------|
| AND gate | LIF, threshold=2 | 1 |
| OR gate | LIF, threshold=1 | 1 |
| NOT gate | Inhibitory LIF | 1 |
| XOR gate | LIF + mutual inhibition | 1 |
| SR latch (1 bit) | 2 mutually inhibiting LIF | 2 |
| Register (8-bit) | 8 SR latches | 16 |
| Full adder | 2 XOR + 2 AND + 1 OR | 5 per bit |
| 8-bit ADD | Ripple-carry chain | 41 |
| 8-bit CMP | Subtract + sign | 24 |
| Sort (N=9) | Bubble comparisons | ~216 |

This demonstrates that spiking networks are **Turing-complete**.
The spike-based ALU operates on the same substrate as learning
and perception — enabling architectures where computation,
memory, and adaptation share a single neuromorphic fabric.

Reference: Plana et al. 2022 (SpiNNaker spike-based logic).